# 03 — CamemBERT QA : Question-Answering Extractif

Ce notebook évalue le modèle `illuin-technology/camembert-base-fquad` pour
la tâche de question-réponse extractive en français :
1. Chargement et test du modèle
2. Évaluation sur des paires question/contexte/réponse attendue
3. Analyse des scores de confiance
4. Mesure de la latence

In [ ]:
import sys, time
sys.path.insert(0, '..')

from transformers import pipeline
import numpy as np

## 1. Chargement du modèle

In [ ]:
qa_pipeline = pipeline(
    "question-answering",
    model="illuin-technology/camembert-base-fquad",
    tokenizer="illuin-technology/camembert-base-fquad"
)
print("✓ Modèle CamemBERT-QA chargé")

## 2. Tests sur des paires question/contexte

In [ ]:
test_cases = [
    {
        "question": "Qu'est-ce que l'apprentissage supervisé ?",
        "context": "L'apprentissage supervisé est une technique d'apprentissage automatique où le modèle apprend à partir de données étiquetées. Chaque exemple d'entraînement contient une entrée et la sortie attendue. Le modèle ajuste ses paramètres pour minimiser l'erreur entre sa prédiction et la réponse correcte.",
        "expected_keywords": ["apprentissage", "données étiquetées", "modèle"]
    },
    {
        "question": "Combien de couches a un perceptron multicouche ?",
        "context": "Le perceptron multicouche (MLP) est un réseau de neurones composé d'au moins trois couches : une couche d'entrée, une ou plusieurs couches cachées, et une couche de sortie. Chaque neurone est connecté à tous les neurones de la couche suivante.",
        "expected_keywords": ["trois", "couches"]
    },
    {
        "question": "Quel est le rôle de la fonction d'activation ?",
        "context": "La fonction d'activation introduit de la non-linéarité dans le réseau de neurones. Sans elle, le réseau ne pourrait modéliser que des relations linéaires. Les fonctions courantes incluent ReLU, sigmoid et tanh. ReLU est la plus utilisée dans les réseaux profonds modernes.",
        "expected_keywords": ["non-linéarité", "fonction"]
    }
]

for tc in test_cases:
    result = qa_pipeline(question=tc["question"], context=tc["context"])
    answer = result["answer"]
    score = result["score"]
    
    # Vérifier si au moins un mot-clé attendu est dans la réponse
    has_keyword = any(kw.lower() in answer.lower() for kw in tc["expected_keywords"])
    status = "✓" if has_keyword else "✗"
    
    print(f"\n{status} Q: {tc['question']}")
    print(f"  Réponse  : {answer}")
    print(f"  Confiance: {score:.4f}")
    print(f"  Mots-clés attendus: {tc['expected_keywords']}")

## 3. Analyse du seuil de confiance

In [ ]:
# Question hors contexte — le score devrait être bas
out_of_scope = qa_pipeline(
    question="Quelle est la capitale de la France ?",
    context="Le perceptron multicouche est un réseau de neurones composé de plusieurs couches."
)
print(f"Question hors contexte :")
print(f"  Réponse  : {out_of_scope['answer']}")
print(f"  Confiance: {out_of_scope['score']:.4f}")
print(f"  Seuil EduAI (0.25) : {'Rejeté ✓' if out_of_scope['score'] < 0.25 else 'Accepté ✗'}")

# Collecter les scores pour les cas valides
valid_scores = []
for tc in test_cases:
    r = qa_pipeline(question=tc["question"], context=tc["context"])
    valid_scores.append(r["score"])

print(f"\nScores des réponses valides :")
print(f"  Min : {min(valid_scores):.4f}")
print(f"  Max : {max(valid_scores):.4f}")
print(f"  Moy : {np.mean(valid_scores):.4f}")

## 4. Benchmark de latence

In [ ]:
N_RUNS = 20
latencies = []

ctx = test_cases[0]["context"] * 3  # Contexte plus long (~512 tokens)

for _ in range(N_RUNS):
    start = time.perf_counter()
    qa_pipeline(question=test_cases[0]["question"], context=ctx)
    latencies.append(time.perf_counter() - start)

print(f"Latence CamemBERT-QA ({N_RUNS} runs, contexte ~{len(ctx)} chars) :")
print(f"  Moyenne : {np.mean(latencies)*1000:.1f} ms")
print(f"  Médiane : {np.median(latencies)*1000:.1f} ms")
print(f"  P95     : {np.percentile(latencies, 95)*1000:.1f} ms")

## Conclusion

- CamemBERT-FQuAD extrait des réponses pertinentes en français
- Le score de confiance permet de filtrer les réponses hors contexte (seuil 0.25)
- La latence reste acceptable pour une utilisation interactive (~200-500 ms sur CPU)